In [1]:
import pandas as pd
import itertools
from scipy.stats import spearmanr
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

# DATA

In [8]:
cols = ['sample', 'SITE_ID', 'offset_rda', 'marker', 'trait', 
       'POP_SITE', 'POP_ID', 'SET', 'mean', 'sd', 'n', 'group', 'train_group']

In [9]:
df1 = pd.read_csv("18_garden_clusters.tsv", sep="\t", header=0)
# only results for combined 
df1[["group","train_group"]].drop_duplicates()
df1 = df1[cols].reset_index(drop=True)
df1.head()

,sample,SITE_ID,offset_rda,marker,trait,POP_SITE,POP_ID,SET,mean,sd,n,group,train_group
0,2209_PR,PR,0.340529,1000,Height,2209_PR,2209,TEST,980.525258,302.158899,4.0,West,Central
1,325_PR,PR,0.378512,1000,Height,325_PR,325,TEST,1239.891145,121.743290,8.0,Central,Central
2,4351_PR,PR,0.278826,1000,Height,4351_PR,4351,TEST,1423.598605,68.068593,4.0,Central,Central
3,4420_PR,PR,0.447724,1000,Height,4420_PR,4420,TEST,678.153897,154.272486,7.0,West,Central
4,6805_PR,PR,0.339265,1000,Height,6805_PR,6805,TEST,1353.974749,229.855027,7.0,East,Central


In [10]:
df2 = pd.read_csv("../951_garden_plot/01B_garden_compare_rda.tsv", sep="\t", header=0)
df2["train_group"] = "combined"
df2 = df2[df2["marker"] == "1000"].reset_index(drop=True)
df2 = df2[cols].reset_index(drop=True)
df2.head()

,sample,SITE_ID,offset_rda,marker,trait,POP_SITE,POP_ID,SET,mean,sd,n,group,train_group
0,2209_PR,PR,0.075343,1000,Height,2209_PR,2209,TEST,980.525258,302.158899,4.0,West,combined
1,325_PR,PR,0.309719,1000,Height,325_PR,325,TEST,1239.891145,121.743290,8.0,Central,combined
2,4351_PR,PR,0.168758,1000,Height,4351_PR,4351,TEST,1423.598605,68.068593,4.0,Central,combined
3,4420_PR,PR,0.220976,1000,Height,4420_PR,4420,TEST,678.153897,154.272486,7.0,West,combined
4,6805_PR,PR,0.242502,1000,Height,6805_PR,6805,TEST,1353.974749,229.855027,7.0,East,combined


In [11]:
df = pd.concat([df1, df2], ignore_index=True)
df = df[df["SET"] == "TEST"].reset_index(drop=True)
print(df.shape)
df.head()

(5649, 13)


,sample,SITE_ID,offset_rda,marker,trait,POP_SITE,POP_ID,SET,mean,sd,n,group,train_group
0,2209_PR,PR,0.340529,1000,Height,2209_PR,2209,TEST,980.525258,302.158899,4.0,West,Central
1,325_PR,PR,0.378512,1000,Height,325_PR,325,TEST,1239.891145,121.743290,8.0,Central,Central
2,4351_PR,PR,0.278826,1000,Height,4351_PR,4351,TEST,1423.598605,68.068593,4.0,Central,Central
3,4420_PR,PR,0.447724,1000,Height,4420_PR,4420,TEST,678.153897,154.272486,7.0,West,Central
4,6805_PR,PR,0.339265,1000,Height,6805_PR,6805,TEST,1353.974749,229.855027,7.0,East,Central


# Get paiwise correlations

In [12]:
df_sub = df[(df["SITE_ID"] == "ML") & (df["trait"] == "Height")].reset_index(drop=True)
df_sub.head()

,sample,SITE_ID,offset_rda,marker,trait,POP_SITE,POP_ID,SET,mean,sd,n,group,train_group
0,1329_ML,ML,0.213509,1000,Height,1329_ML,1329,TEST,1435.746514,165.025251,3.0,East,East
1,1528_ML,ML,0.139633,1000,Height,1528_ML,1528,TEST,1424.030354,75.498344,3.0,East,East
2,1530_ML,ML,0.336305,1000,Height,1530_ML,1530,TEST,976.829721,155.349069,3.0,East,East
3,1531_ML,ML,0.159446,1000,Height,1531_ML,1531,TEST,1186.591020,130.000000,3.0,East,East
4,1534_ML,ML,0.081893,1000,Height,1534_ML,1534,TEST,1367.178745,110.151411,3.0,East,East


In [13]:
# Get all unique factors and their pairwise combinations

def get_pearsons_corr(dataset):
    factors = dataset["train_group"].unique()
    pairs = list(itertools.combinations(factors, 2))
    pairs
    
    results = []
    
    for f1, f2 in pairs:
        v1 = dataset.loc[dataset["train_group"] == f1, "offset_rda"].values
        v2 = dataset.loc[dataset["train_group"] == f2, "offset_rda"].values
    
        # Only compare if both have the same number of observations
        n = min(len(v1), len(v2))
        rho, pval = pearsonr(v1[:n], v2[:n])
        rho, pval = pearsonr(v1, v2)
        results.append({"train_group1": f1, "train_group2": f2, "r": rho, "p": pval, "n" : n})
    
    pairwise_corr = pd.DataFrame(results)
    return(pairwise_corr)

In [14]:
get_pearsons_corr(df_sub).head()

,train_group1,train_group2,r,p,n
0,East,Central,0.885167,3.405238e-14,40
1,East,combined,0.980135,2.621057e-28,40
2,Central,combined,0.819248,1.030673e-10,40


# Loop

In [15]:
R = []
for site in ["PR","ML","CH","AC"]:
    for trait in df['trait'].drop_duplicates().values:
        df_sub = df[(df["SITE_ID"] == site) & (df["trait"] == trait)].reset_index(drop=True)

        df_corrs = get_pearsons_corr(df_sub)
        df_corrs['SITE_ID'] = site
        df_corrs['trait'] = trait
        R.append(df_corrs)
dR = pd.concat(R, ignore_index=True)
dR.head()

,train_group1,train_group2,r,p,n,SITE_ID,trait
0,Central,West,0.858259,2.066011e-08,26.0,PR,Height
1,Central,combined,0.729918,2.315485e-05,26.0,PR,Height
2,West,combined,0.946370,2.837389e-13,26.0,PR,Height
3,Central,West,0.864056,1.291522e-08,26.0,PR,Biomass_Increment
4,Central,combined,0.759509,6.809436e-06,26.0,PR,Biomass_Increment


### Saving

In [16]:
dR.to_csv("22_compare_offsets.tsv", sep="\t", header=True, index=False)